In [1]:
%load_ext autoreload
%autoreload 2

# 2 Angle Classifier

This notebook should be run after having completed the `1 - usability tagger.ipynb` notebook and the `apply_bintag.ipynb` notebook. These two notebooks are responsible for cleaning up the dataset and help to avoid letting the models learn on images of interior elements, car accesoires, odometers, placeholder images etc...

The goal of this notebook is to predict the angle at which an image is taken and see if this is usefull to finetune models on. The reasoning behind this are visual cues often shared among different models of the same brand: 

e.g. BMW has a few distinctive visual cues in their design language of cars: 
- Frontside: Kidney grill - an element that has been introduced on the BMW 303 in 1933 and has known various redesigns (https://www.bmw.com/en/design/the-bmw-kidney-grille-through-time.html).
- Sideview: Hoffmeister kink - A subtle curve in the C-pillar (not exclusively but predominantly used by BMW: https://en.wikipedia.org/wiki/Hofmeister_kink )

e.g. Volvo:
- Frontside: Thor's hammer headlights: Used in modern cars the DRL's are inspired by Swede's folklore and dubbed the "Thor's Hammer Headlights" (https://www.matthewsvolvosite.com/thors-hammer-headlights/) 

... 

The aim of this notebook is to provide an estimate of the angle at which a picture is taken. In the next phase (Brand Prediction) We'll compare if training models on specific angles are more precise than models trained on all angles at once. The idea is that a model will pick up these visual cues by focussing on only a specific angle more effectively than seeing the whole dataset (this is to be investigated)

Codewise this notebook will focus on implementing new elements that might be needed later: 
1) Out of Core learning: using some kind of batch processing of the image data (Hard requirement for processing 4million images at the brand identification stage)
2) Data augmentation: adding a few functions to add more variance in the data (Hard requirement for predicting low-occuring models e.g. Ford Anglia...)

In [2]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import OrdinalEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, cohen_kappa_score
import os
import sys
import re
import random
import uuid
from tqdm import tqdm


sys.path.append('../../utils')
import config_handling as conf
#from database import Database
from file_io import make_absolute_path
import implot
import cnn_helpers

2025-03-03 23:07:35.181371: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
#TODO: this is a better way of handling the db and basedir from conf than in phase one, apply it in other places too.
basedir, db  = conf.applyconf('../../config/automotive.conf.ini')

Connection established


In [4]:
augment_base = os.path.join(basedir, 'augmentated data', 'angle phase')

In [5]:
# Utilities that are only needed in the Angle Tagging phase: 
def augmentate_data(df): 
    """
        Takes a dataframe and returns a list of new rows so that every angle has an equal 
        amount of instances as defined by the INSTANCES constant. 

        parameter: df (pandas dataframe) with angle column

        returns list of rows matching the columns in df
    """
    new_rows = []
    for angle in df.angle.unique():
        relevant_angles = df.query(f'angle=="{angle}"')
        amount_to_augment = INSTANCES - len(relevant_angles)
        for _ in tqdm(range(amount_to_augment)):
            base_image = relevant_angles.sample(1)
            augmented = augment(base_image)
            new_rows.append(augmented)
    return new_rows

def augment(sampled_df): 
    row = sampled_df.iloc[0].copy()
    impath = row['abs_path']
    uuid_v4 = str(uuid.uuid4())
    new_name = uuid_v4 + '.' + impath.split('.')[-1]
    coords = (row['yolobox_top_left_x'], row['yolobox_top_left_y'], row['yolobox_bottom_right_x'], row['yolobox_bottom_right_y'])
    image = cv2.imread(impath)
    augmented_path = os.path.join(augment_base, new_name)
    os.makedirs(os.path.dirname(augmented_path), exist_ok=True)

    base_action = random.randint(0,100)
    #80% of images has only ONE mutation
    #15% has TWO mutations
    # 5% of images has THREE mutatations applied. 
    mutation_options = [0, 1, 2]
    if base_action < 85: 
        mutations = random.sample(mutation_options, 1)         #pick ONE mutation
    elif base_action < 95: 
        mutations = random.sample(mutation_options, 2)         #pick TWO mutations
    else:
        mutations = random.sample(mutation_options, len(mutation_options))              #Perform ALL THREE mutations in random order.
    modded_img = image.copy()
    for mutation in mutations: 
        if mutation == 0: 
            modded_img = cnn_helpers.augm_channel_shuffle(modded_img)
        elif mutation == 1:
            sigma = random.randint(40, 80)
            modded_img = cnn_helpers.augm_add_random_noise(modded_img, 0, sigma)
        else:
            stretch = random.uniform(0.8, 1.3)
            modded_img, coords = cnn_helpers.augm_stretch_image_and_bbox(modded_img, coords, stretch)
    #print(type(modded_img))
    #print(modded_img.shape())
    cv2.imwrite(augmented_path, modded_img)
    row['abs_path'] = augmented_path
    row['yolobox_top_left_x'] = coords[0]
    row['yolobox_top_left_y'] = coords[1]
    row['yolobox_bottom_right_x'] = coords[2]
    row['yolobox_bottom_right_y'] = coords[3]
    return row  


def learn_angle(X, y, validation_split, bbox):
    """I will not make this a reusable component for other learning tasks, this is only for angle learning. 
    Might use other parameters/ model sequences for the actual brand and or model predictions. Since it is limited
    to angle learning, it'll not be made part of the utilities. 

    X = dataframe of training data
    y = dataframe with targets
    note that X and y should be shuffled before passing as arguments.
    validation_split = float (0 to 1). how the data should be split
    bbox = bool

    returns; nothing.
    ouput = model dump per epoch. 
    """
    #Note: I'm comfortable with this way of splitting, remember that the datframe
    # was shuffled at random and that X_train was a perfectly even distributed class.
    # Yes, there will be some inequality, but this is not of any significance. 
    train_data = X[:int(1-validation_split * len(X))]
    val_data = X[int(1-validation_split * len(X)):]
    y_train_data = y[:int(1-validation_split * len(X))]
    y_val_data = y[int(1-validation_split * len(X)):]

    assert(len(train_data)+ len(val_data) == len(X))
    assert(len(y_train_data) ==  len(train_data))


    # Deep learning part here: implement the plateau and early stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=6, 
        restore_best_weights=True
    )
    lr_scheduler = ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=3, 
        min_lr=1e-6, 
        verbose=1
    )

    e = '{epoch:02d}'
    model_checkpoint = ModelCheckpoint(
        filepath=os.path.join(model_dest_dir, f'model_angles-bbox={bbox}-epoch={e}.keras'),
        save_freq=epoch_checkpoint,
        save_best_only=False,
        verbose=0
    )


    # Angle classifier with CNN
    angle_model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(shape, shape, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(len(ANGLES), activation='softmax')
    ])
    angle_model.compile(
        optimizer=Adam(learning_rate=0.001), 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )
    #Do just in time preprocessing of the images: yields both the numpy array and targets!
    train_gen = cnn_helpers.image_generator(
        batch_size=batch_size, 
        data_frame=train_data, 
        bboxs=bboxs, 
        shape=shape, 
        y_train_encoded=y_train_data
    )
    #Split validation separately
    val_gen = cnn_helpers.image_generator(
        batch_size=batch_size, 
        data_frame=val_data, 
        bboxs=bboxs, 
        shape=shape, 
        y_train_encoded=y_val_data
    )

    angle_model.fit(
        train_gen,
        epochs=max_epoch, 
        steps_per_epoch=len(train_data) // batch_size,
        batch_size=batch_size, 
        validation_data=val_gen,
        validation_steps = len(val_data) // batch_size,
        callbacks=[early_stopping, lr_scheduler, model_checkpoint] 
    )
    #Okay, the callback principle is really cool.
    # You just put a bunch of assigned variables in there that are mapped to some part of the KERAS API and it automagically handles everything for you. 

## 2.1 Constants:

In [6]:
#In phase 1 we decided to have a cutoff point at 0.9pct. Put it here as a constant
BIN_TRESHOLD = 0.9
#For each tag, how many instances will we shown as part of the training data (AFTER SPLITTING)
INSTANCES = 2500
#Put the angles in order: nicer to work with later on in CM it is structured logically: 
ANGLES = ["front", "frontleft", "left", "rearleft", "rear", "rearright", "right", "frontright"]
#Random state not used: this notebook is ran a few times untill 2.9; varying random state is good in this case.
#RS = 42

## 2.2 Getting tagged data for training purposes:

In [7]:
get_traindata_query = """
    SELECT * 
    FROM angletags
    JOIN images ON images.id = angletags.image_id
    WHERE angletags.angle <> 'crappy'
    """
traindata = db.execute_query(get_traindata_query)
traindf = pd.DataFrame(traindata)
traindf = make_absolute_path(traindf, basedir, 'image_path', 'abs_path', True)
traindf.sample(10)

,image_id,angle,manual_annotation,pk,id,listing_id,processed,use_image,certainty_of_outside_yolo,yolobox_top_left_x,yolobox_top_left_y,yolobox_bottom_right_x,yolobox_bottom_right_y,area,confidence,abs_path
5682,838149,rear,1,15240,838149,53671,1,0,None,NaN,NaN,NaN,NaN,NaN,NaN,/home/frederic/Documents/automotive_image_data...
6336,625135,frontright,1,17112,625135,39742,1,1,None,23.0,36.0,742.0,507.0,0.802967,0.872853,/home/frederic/Documents/automotive_image_data...
6045,343574,rearright,1,16262,343574,21546,1,1,None,113.0,111.0,640.0,415.0,0.379428,0.953717,/home/frederic/Documents/automotive_image_data...
2435,891762,front,1,6331,891762,57373,1,0,None,NaN,NaN,NaN,NaN,NaN,NaN,/home/frederic/Documents/automotive_image_data...
6041,64626,rear,1,16253,64626,4405,1,1,None,173.0,256.0,597.0,551.0,0.296456,0.854144,/home/frederic/Documents/automotive_image_data...
1645,1091233,right,1,4214,1091233,70470,1,1,None,21.0,176.0,698.0,480.0,0.488570,0.734214,/home/frederic/Documents/automotive_image_data...
5141,1081038,rear,1,13856,1081038,69868,1,1,None,82.0,158.0,356.0,380.0,0.254717,0.867762,/home/frederic/Documents/automotive_image_data...
748,334976,rearright,1,1978,334976,20992,1,1,None,32.0,86.0,734.0,538.0,0.753398,0.864742,/home/frederic/Documents/automotive_image_data...
303,964564,rearleft,1,781,964564,62532,1,1,None,60.0,119.0,724.0,490.0,0.583442,0.710083,/home/frederic/Documents/automotive_image_data...
3328,33065,rearright,1,8795,33065,2336,1,1,None,23.0,57.0,731.0,545.0,0.818638,0.645208,/home/frederic/Documents/automotive_image_data...


In [8]:
traindf.angle.value_counts()

angle
frontleft     1491
frontright    1127
rearright     1058
rearleft      1014
front          906
rear           788
right          493
left           484
Name: count, dtype: int64

There's some class imbalance here (frontleft is over 3 times more prevalent than shots taken from the left). We'll use oversampling techniques on all classes to show the model `INSTANCES` amount of instances per class. In an attempt to improve generalization we'll add small image manipulation methods to an image that is not shown for the first time (e.g. gaussian noice, warping, stretching....) WE DO NOT FLIP OR MIRROR the image, this would mean a different antletag would be needed. (You could for instance flip an image LEFT to RIGHT and then use the other tag, but we'll not be doing this!)

Before augmenting, we need to apply a solid splitting strategy to avoid feature leaking: 

## 2.3 Train test split.
Use a stratified split based on the angle to use 20% of the data as unseen testdata. This is done based on the *angle* label.

In [9]:
#no random state here, will run this notebook a few times
X_train, X_test = train_test_split(traindf, test_size=0.2, stratify=traindf.angle)


In [10]:
assert (len(traindf) == len(X_train) + len(X_test))

`X_test` are not touched for learning; we stash them away and look back to them after training a CNN. NO augmentation will be done on X_test. Notice that we do not have our targets yet. It's a bit easier to work with like this untill I have my classes balanced out using augmentation.

## 2.4 Applying data augmentation and getting balanced classes: 
I'm limited with memory and don't want to have the augmented images in memory. In stead I'll use the SSD (M2 drive - so fast IO) to store these augmented pictures. (Sacrifice some storage, to gain free memory). The overhead is the IO speed of the M2 SSD vs IO speed of RAM - which is a significant penalty, but I can live with this. 

Augmented images will be stored using a UUIDV4 (almost guaranteed to be unique: see pigeonhole principle) so I'm not worried about overwriting data. augmented images will find a nice welcoming home in a dedicated folder that I can send to oblivition after the notebook is done - these are not needed further down the lifecyle of this project.

The content of the `X_test` dataframe will not be augmented, the `X_train` dataframe will be augmented so that all classes are equally present. This will be stored in a new dataframe at the end of section 2.4

In [11]:
#otherwise tf keeps complaining when adding gaussian noise
cnn_helpers.system_override()
device = cnn_helpers.system_pick_device()

System override applied - check if GPU is detected
Using GPU for deep learning.


2025-03-03 23:07:51.549757: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-03 23:07:53.961125: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-03 23:07:53.961212: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [13]:
cnn_helpers.wipe_folder(augment_base)

In [ ]:

augmented_rows = augmentate_data(X_train)
augmented_df = pd.concat([X_train, pd.DataFrame(augmented_rows)], axis=0)
augmented_df = augmented_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
#finally prepare the augmented_df by cating the bbox fields to int and setting nan to sentinel -1
augmented_df = cnn_helpers.cast_bbox_values(augmented_df)

In [ ]:
augmented_df.angle.value_counts()
#augmentation was good.

## 2.5 Assigning Train/test variables
I kept my labels part of the dataframe during the data augmentation phase, with that out of hte way, time to go back to standard practice. We'll not be using `train_test_split()` method here as we already have this split done. 

In [ ]:
y_train = augmented_df['angle']
X_train = augmented_df.drop(columns=['angle'])
y_test = X_test['angle']
X_test = X_test.drop(columns=['angle'])

## 2.6 Label encoding.
We need to convert the `ANGLES` cosntant to  integer labels. Originally this notebook used `LabelEncoder`, however this does not preserve the order we have in `ANGLES`. This order does make it easier to interpret the confusionmatrix at the end. To overcome this nuissance we use `ordinalencoder`. More info on this issue: https://github.com/scikit-learn/scikit-learn/issues/12086 

In [ ]:
angles_nested = [[angle] for angle in ANGLES]
ordinal_encoder = OrdinalEncoder(categories=[ANGLES])
ordinal_encoder.fit(angles_nested)
y_train_encoded = ordinal_encoder.transform([[label] for label in y_train]).flatten().astype('int')
y_test_encoded = ordinal_encoder.transform([[label] for label in y_test]).flatten().astype('int')

## 2.7 Training a CNN
In phase one epochs weren't implemented in an optimal way, apparently you can mid-epoch decide to dump and store the result. Python can also self-monitor how much improvement is made per epoch. So it can early stop. These techniques are part of callbacks API (earlystopping and ModelCheckpoint)
https://machinelearningmastery.com/check-point-deep-learning-models-keras/

These techniques will be used in phase 2

In [ ]:
#how large will the image be to train on: If you change it here, whole script is updated
shape = 256

#Use or Not use BBOX? In phase 1 it wasn't worth the effort, what about angle tagging?? 
# I expect it to be useful here as cutting an image of does not make i harder to identify
# and a cut off image does not render it 'invalid' for this task.
bboxs = [True, False]

#At how many epoch(stps) should a dump of the model be made (Checkpoints)
epoch_checkpoint = 5 
#What's the maximum amount of epochs to go over before killing the learing process.
max_epoch = 100

#Batch size
batch_size = 32

#number of classes: 
numclasses = len(ANGLES)

#For online learning we presplit our X_train in a new train and validation set, define ratio here: 
validation_split = 0.2

#where to save (checkpoint iterations of ) the model?
model_dest_dir = os.path.join(os.getcwd(), '..', '..', 'models', 'angle_models')
os.makedirs(model_dest_dir, exist_ok=True)


In [ ]:
for bbox in bboxs:
    learn_angle(X_train, y_train_encoded, validation_split, bbox)

## 2.7 Asserting model performance

Let's process the original testdata of images for each of the eight angles; this batch is small enough to fit in memory, so we run it in a single go. For this we'll be revisiting the variables `X_test` and `y_test` created in step 2.3. 

We have two variants on the same model: one with and one without BoundingBox. The images need to be preprocessed in the same way to assess performance. 

Just out of interest we'll also use non-bound images on the bounded model and vice versa; in total we'll be generating four confusion matrices: 

1) Model With BBOX + Images with BBOX
2) Model without BBOX + Images without BBOX
3) Model With BBOX * Images without BBOX
4) Model without BBOX + Images with BBOX

The two final models are simply the models with the highest Epoch count; one with BBOX to true in the filename and one without. Since epochs are time-sensitively created (epoch 2 is created later than epoch 1); we can simply take the 'most recent created file' for BBOX TRUE and for BBOX FALSE, those are automatically the best models. 

In [ ]:
models = os.listdir(model_dest_dir)
model_paths = [os.path.join(model_dest_dir, f) for f in models if os.path.isfile(os.path.join(model_dest_dir, f))]
sorted_models = sorted(model_paths, key=os.path.getctime)


In [ ]:
X_test = cnn_helpers.cast_bbox_values(X_test)
decoded_labels = ordinal_encoder.inverse_transform([[i] for i in range(8)]).flatten()
fig, axs = plt.subplots(2, 2, figsize=(12, 12))
axs = axs.flatten()
idx = 0
metrics_dict = {}
for crop_image in bboxs:
    X_test_images = []
    for _, row in X_test.iterrows():
        coords = [row['yolobox_top_left_x'], row['yolobox_top_left_y'], row['yolobox_bottom_right_x'], row['yolobox_bottom_right_y']]
        image = cnn_helpers.preprocess_image(row['abs_path'], crop_image, coords, shape)
        X_test_images.append(image)
    X_test_images = np.array(X_test_images)
    for use_cropped_model in bboxs:
        matching_models = [f for f in sorted_models if f'bbox={use_cropped_model}-' in f]
        model = load_model(matching_models[-1])
        predicts = model.predict(X_test_images)
        predicts = np.argmax(predicts, axis=1)
        decoded_actuals = ordinal_encoder.inverse_transform(y_test_encoded.reshape(-1, 1)).flatten()
        decoded_predicts = ordinal_encoder.inverse_transform(predicts.reshape(-1, 1)).flatten()
        confusion_on_y_test = pd.DataFrame(
            {
                'actuals': decoded_actuals,
                'predicts': decoded_predicts
            }
        )

        #confusion_on_y_test.sample(5)
        cm = confusion_matrix(confusion_on_y_test['actuals'], confusion_on_y_test['predicts'], labels=decoded_labels)

        metrics_dict[f'image={crop_image}_model={use_cropped_model}'] = {
            'Accuracy': accuracy_score(decoded_actuals, decoded_predicts),
            'Macro F1 Score': f1_score(decoded_actuals, decoded_predicts, average='macro'),
            'Weighted F1 Score': f1_score(decoded_actuals, decoded_predicts, average='weighted'),
            'Cohen\'s Kappa': cohen_kappa_score(decoded_actuals, decoded_predicts)
        }

        # Display confusion matrix using matplotlib
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=decoded_labels)
        disp.plot(ax=axs[idx], cmap='Blues', colorbar=False)
        axs[idx].set_title(f"CM: imaged cropped = {crop_image} -- cropped model = {use_cropped_model}")
        axs[idx].set_xlabel("Predictions")
        axs[idx].set_ylabel("Actuals")
        idx +=1

plt.tight_layout()


It's quite apparent that the worst combination of cropping factors is when the image given to predict is not cropped. It doesn't seem to matter a lot whether or not the image is trained on a cropped image or not, the margin of error is almost always quite equal. 

An interesting observations in the well-performing combinations is that the model most often struggles with opposite views (e.g. Actual = Rear, Predict = Front). 

In [ ]:
metrics = ['Accuracy', 'Macro F1 Score', 'Weighted F1 Score', "Cohen's Kappa"]
labels = list(metrics_dict.keys())  # These are the combinations of image and model
metric_values = {metric: [] for metric in metrics}

for label in labels:
    for metric in metrics:
        metric_values[metric].append(metrics_dict[label][metric])

fig, axs = plt.subplots(2, 2, figsize=(12, 12))
axs = axs.flatten()

for idx, metric in enumerate(metrics):
    bars = axs[idx].bar(labels, metric_values[metric], color='skyblue')
    axs[idx].bar(labels, metric_values[metric])
    axs[idx].set_title(f'{metric} Comparison')
    axs[idx].set_ylabel(metric)
    axs[idx].set_xticklabels(labels, rotation=45, ha="right")
    axs[idx].set_ylim([0, 1])
    for bar in bars:
        yval = bar.get_height()
        axs[idx].text(bar.get_x() + bar.get_width() / 2, yval + 0.02,  # Add a small offset to place the text above the bar
                      f'{yval:.5f}', ha='center', va='bottom', fontsize=10)  # Rounded to 4 decimal places



plt.tight_layout()

The model that seems to perform the best, is a model where uncropped images where used to train, the predictions are the best when the image to perform a prediction on is cropped to the Boundingbox found by YOLO. The difference is however very very small.

## 2.8 Deciding on a model and training on full augmented dataset.
The current results are ambiguous, sometimes these are in favor of a model that was trained with or without cropping the image in the training phase. 
** Accuracy metric **

- Run 1
    - True-True: 0.91446
    - True-False: 0.91582
- Run 2
    - True-True: 0.89477
    - True-False: 0.89409
- Run 3
    - True-True: 0.
    - True-False: 0.
- Run 4
    - True-True: 0.
    - True-False: 0.
- Run 5
    - True-True: 0.
    - True-False: 0.
- Run 6
    - True-True: 0.
    - True-False: 0.



## 2.9 Using the fully tagged dataset for training a final model.

In [ ]:
traindf

In [ ]:
#augment based on traindf:
augment_df_final = 
X = traindf.drop(columns=['angle'])
y = traindf['angle']
crop_in_training = True